In [ ]:
# %pip install patool tensorflow pillow opencv-python scipy numpy matplotlib scikit-learn

In [ ]:
# import patoolib
# patoolib.extract_archive("dataset_processed.zip",outdir="./content/")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np


np.random.seed(42)
tf.random.set_seed(42)

IMG_SIZE = 224
BATCH_SIZE = 16

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet.preprocess_input,
    rotation_range=5,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    brightness_range=[0.8, 1.2]
)

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet.preprocess_input
)

train_data = train_datagen.flow_from_directory(
    'content/dataset_processed/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    'content/dataset_processed/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
from sklearn.utils import class_weight

y_train = train_data.classes

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

classes = np.unique(y_train)
class_weights = dict(zip(classes, weights))

print("Class Weights:", class_weights)

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Congelar base (primeira fase)
for layer in base_model.layers:
    layer.trainable = False

# Cabeça nova (classificador)
x = base_model.output
avg = keras.layers.GlobalAveragePooling2D()(base_model.output)
outputs = keras.layers.Dense(train_data.num_classes, activation="softmax")(avg)

# x = base_model.output
# x = keras.layers.GlobalAveragePooling2D()(x)
# x = keras.layers.BatchNormalization()(x)
# x = keras.layers.Dense(128, activation='relu')(x)
# x = keras.layers.Dropout(0.5)(x)
# outputs = keras.layers.Dense(train_data.num_classes, activation="softmax")(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilar
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=["accuracy"]
)

# Treinar (fase 1)
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=5,
    class_weight=class_weights
)

In [ ]:
# Fine-tuning
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR menor
    loss='categorical_crossentropy',
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_data,
    validation_data=test_data,
    epochs=5,
    class_weight=class_weights
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 🔹 1. Garantir ordem correta
test_data.reset()

# 🔹 2. Predições
y_pred_probs = model.predict(test_data, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 3. Labels reais
y_true = test_data.classes

# 🔹 4. Nomes das classes
class_names = list(test_data.class_indices.keys())

# 🔹 5. Matriz de confusão
cm = confusion_matrix(y_true, y_pred)

cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(
    cmap='Blues',
    ax=ax,
    xticks_rotation=45,
    values_format='.2f'
)

plt.title("Matriz de Confusão Normalizada")
plt.grid(False)
plt.tight_layout()
plt.savefig("confusion_matrix_ef.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))